# Tier 2 — Live API Integration Tests (~$0.10-0.50)

These tests make real API calls. **Estimated total cost: $0.05-0.10**.

**Prerequisites:**
- `.env` file with `ANTHROPIC_API_KEY` set
- `config.json` (or `config-base.json`) with Pricing data
- `pip install nest_asyncio` (for Jupyter async support)

**Run All** with `Kernel → Restart & Run All` for a full pass.

In [ ]:
# Setup — load real config, init provider, display cost warning
import sys
import os
import json
from pathlib import Path

nb_dir = str(Path(".").resolve())
if nb_dir not in sys.path:
    sys.path.insert(0, nb_dir)

from test_helpers import (
    bootstrap_pricing, assert_pass, print_summary, run_async,
    UsageResult, SessionAccumulator, BufferedChannel,
)

# Load .env
root = Path(".").resolve().parent
env_path = root / ".env"
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, _, val = line.partition("=")
            os.environ.setdefault(key.strip(), val.strip())

bootstrap_pricing()

# Verify API key is available
api_key = os.environ.get("ANTHROPIC_API_KEY", "")
assert_pass(bool(api_key), "ANTHROPIC_API_KEY is set")

# Init real Anthropic provider
from micro_x_agent_loop.providers.anthropic_provider import AnthropicProvider

provider = AnthropicProvider(api_key=api_key)
MODEL = "claude-haiku-4-5-20251001"  # Cheapest model for tests

print("\n" + "=" * 60)
print("  ⚠️  COST WARNING: These tests make real API calls.")
print(f"  Model: {MODEL}")
print("  Estimated total cost: ~$0.05-0.10")
print("=" * 60)
print("\nSetup complete.")

## 2.1 Live API — SessionAccumulator Fields
*Covers: [MANUAL-TEST-pricing.md](../documentation/docs/testing/MANUAL-TEST-pricing.md) Test 1.1*

In [ ]:
# 2.1 — Send one prompt, verify SessionAccumulator fields > 0
print("2.1 Live API — SessionAccumulator Fields")

acc = SessionAccumulator(session_id="test-2.1")

messages = [{"role": "user", "content": "Say exactly: hello"}]
system_prompt = "You are a test assistant. Be very brief."

msg, tool_blocks, stop_reason, usage = run_async(
    provider.stream_chat(
        model=MODEL,
        max_tokens=100,
        temperature=0,
        system_prompt=system_prompt,
        messages=messages,
        tools=[],
    )
)

acc.add_api_call(usage, call_type="main", turn_number=1)

print(f"  Response: {msg}")
print(f"  Usage: in={usage.input_tokens}, out={usage.output_tokens}, "
      f"cache_read={usage.cache_read_input_tokens}, cache_create={usage.cache_creation_input_tokens}")
print(f"  Cost: ${acc.total_cost_usd:.6f}")

assert_pass(usage.input_tokens > 0, f"input_tokens > 0 ({usage.input_tokens})")
assert_pass(usage.output_tokens > 0, f"output_tokens > 0 ({usage.output_tokens})")
assert_pass(acc.total_cost_usd > 0, f"total_cost > 0 (${acc.total_cost_usd:.6f})")
assert_pass(acc.total_api_calls == 1, "api_calls = 1")
assert_pass(usage.duration_ms > 0, f"duration_ms > 0 ({usage.duration_ms:.0f})")

print("\n2.1 complete.")

## 2.2 Prompt Caching — Cache Read on Turn 2
*Covers: [MANUAL-TEST-prompt-caching.md](../documentation/docs/testing/MANUAL-TEST-prompt-caching.md) Test 2*

In [ ]:
# 2.2 — Send 2 prompts, assert cache_read > 0 on turn 2
print("2.2 Prompt Caching — Cache Read on Turn 2")

# Use a system prompt large enough to be cached (>1024 tokens for Anthropic)
long_system = (
    "You are a helpful test assistant. " * 200
    + "Always respond with exactly one word."
)

messages_t1 = [{"role": "user", "content": "Say: apple"}]

_, _, _, usage1 = run_async(
    provider.stream_chat(
        model=MODEL, max_tokens=50, temperature=0,
        system_prompt=long_system, messages=messages_t1, tools=[],
        prompt_caching_enabled=True,
    )
)
print(f"  Turn 1: in={usage1.input_tokens}, cache_create={usage1.cache_creation_input_tokens}, "
      f"cache_read={usage1.cache_read_input_tokens}")

# Turn 2 — same system prompt, should get cache read
messages_t2 = [
    {"role": "user", "content": "Say: apple"},
    {"role": "assistant", "content": [{"type": "text", "text": "apple"}]},
    {"role": "user", "content": "Say: banana"},
]

_, _, _, usage2 = run_async(
    provider.stream_chat(
        model=MODEL, max_tokens=50, temperature=0,
        system_prompt=long_system, messages=messages_t2, tools=[],
        prompt_caching_enabled=True,
    )
)
print(f"  Turn 2: in={usage2.input_tokens}, cache_create={usage2.cache_creation_input_tokens}, "
      f"cache_read={usage2.cache_read_input_tokens}")

assert_pass(
    usage2.cache_read_input_tokens > 0,
    f"cache_read > 0 on turn 2 ({usage2.cache_read_input_tokens})",
)
assert_pass(
    usage2.cache_read_input_tokens > usage1.cache_read_input_tokens,
    "turn 2 cache_read > turn 1 cache_read",
)

print("\n2.2 complete.")

## 2.3 Session Budget — Exhaustion
*Covers: [MANUAL-TEST-session-budget.md](../documentation/docs/testing/MANUAL-TEST-session-budget.md) Tests 1-2*

In [ ]:
# 2.3 — Configure tiny budget, verify warning/stop via SessionAccumulator
print("2.3 Session Budget — Exhaustion")

from micro_x_agent_loop.constants import SESSION_BUDGET_WARN_THRESHOLD

acc = SessionAccumulator(session_id="test-2.3")
budget_usd = 0.001  # $0.001 — will be exceeded after 1-2 calls

messages = [{"role": "user", "content": "Count from 1 to 10."}]
system_prompt = "You are a test assistant."

exhausted = False
warned = False
turns = 0
max_turns = 5

while turns < max_turns and not exhausted:
    turns += 1
    _, _, _, usage = run_async(
        provider.stream_chat(
            model=MODEL, max_tokens=200, temperature=0,
            system_prompt=system_prompt, messages=messages, tools=[],
        )
    )
    acc.add_api_call(usage, call_type="main", turn_number=turns)

    pct = acc.total_cost_usd / budget_usd
    print(f"  Turn {turns}: cost=${acc.total_cost_usd:.6f}, budget pct={pct:.1%}")

    if pct >= SESSION_BUDGET_WARN_THRESHOLD:
        warned = True
    if acc.total_cost_usd >= budget_usd:
        exhausted = True

    messages.append({"role": "user", "content": f"Now count from {turns*10} to {turns*10+10}."})

assert_pass(exhausted, "budget was exhausted")
assert_pass(warned, "warning threshold was reached")

toolbar = acc.format_toolbar(budget_usd=budget_usd)
print(f"  Toolbar: {toolbar}")
assert_pass("%" in toolbar, "toolbar shows budget percentage")

print("\n2.3 complete.")

## 2.4 Compaction — Fact Recall After Summarization
*Covers: [MANUAL-TEST-compaction-strategy.md](../documentation/docs/testing/MANUAL-TEST-compaction-strategy.md) Test 5*

In [ ]:
# 2.4 — Trigger compaction with low threshold, verify fact recall
print("2.4 Compaction — Fact Recall After Summarization")

from micro_x_agent_loop.compaction import SummarizeCompactionStrategy
from micro_x_agent_loop.providers.anthropic_provider import AnthropicProvider

compaction_provider = AnthropicProvider(api_key=api_key)
compaction_results = []

def on_compaction(usage, before, after, count):
    compaction_results.append({"before": before, "after": after, "count": count})
    print(f"  Compaction: {count} messages, {before:,} → {after:,} est. tokens")

strategy = SummarizeCompactionStrategy(
    provider=compaction_provider,
    model=MODEL,
    threshold_tokens=200,  # Very low to force trigger
    protected_tail_messages=2,
    on_compaction_completed=on_compaction,
)

# Seed a fact in early messages, then add filler to trigger compaction
secret_fact = "The secret code is ZEBRA-42."
filler = "Tell me about the history of computing in great detail. " * 20

messages = [
    {"role": "user", "content": f"Remember this fact: {secret_fact}"},
    {"role": "assistant", "content": f"I will remember: {secret_fact}"},
    {"role": "user", "content": filler},
    {"role": "assistant", "content": "Here is a long response about computing history. " * 30},
    {"role": "user", "content": "What was the secret code I told you earlier?"},
    {"role": "assistant", "content": "Let me check my context."},
]

compacted = run_async(strategy.maybe_compact(messages))

assert_pass(len(compaction_results) == 1, "compaction was triggered")
assert_pass(len(compacted) < len(messages), f"messages reduced: {len(messages)} → {len(compacted)}")

# Check that the secret fact survived in the compacted summary
all_text = " ".join(
    m.get("content", "") if isinstance(m.get("content"), str)
    else " ".join(
        b.get("text", "") for b in m.get("content", []) if isinstance(b, dict)
    )
    for m in compacted
)
assert_pass(
    "ZEBRA" in all_text.upper() or "secret code" in all_text.lower(),
    "secret fact preserved after compaction",
)

print("\n2.4 complete.")

## Summary

In [ ]:
print_summary()